# Yuki XTTS v2 on Colab

Este notebook sobe um servidor HTTP compatível com o engine `x_tts` da Yuki.

In [ ]:
!nvidia-smi
!wget -q -O /content/miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash /content/miniforge.sh -b -p /content/conda
!/content/conda/bin/conda create -y -n xtts python=3.11
!/content/conda/envs/xtts/bin/pip install -q TTS==0.22.0 fastapi uvicorn python-multipart requests nest-asyncio
PYTHON_BIN = '/content/conda/envs/xtts/bin/python'
print('Usando Python do XTTS:', PYTHON_BIN)

In [ ]:
from google.colab import files
import os
os.makedirs('/content/ref_audio', exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Envie o audio de referencia antes de continuar.')
uploaded_name = next(iter(uploaded))
with open('/content/ref_audio/reference.wav', 'wb') as f:
    f.write(uploaded[uploaded_name])
print('Referencia salva em /content/ref_audio/reference.wav')

In [ ]:
%%writefile /content/xtts_server.py
import os
import tempfile
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from TTS.api import TTS
import torch

REF_AUDIO = '/content/ref_audio/reference.wav'
DEFAULT_LANG = 'pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
app = FastAPI()

class TTSRequest(BaseModel):
    text: str
    speaker_wav: str | None = None
    language: str = DEFAULT_LANG

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'xtts_v2', 'device': device}

@app.post('/tts_to_audio')
def tts_to_audio(req: TTSRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail='text is empty')
    ref_audio = req.speaker_wav if req.speaker_wav and os.path.exists(req.speaker_wav) else REF_AUDIO
    fd, out_path = tempfile.mkstemp(suffix='.wav')
    os.close(fd)
    tts.tts_to_file(
        text=req.text,
        speaker_wav=ref_audio,
        language=req.language or DEFAULT_LANG,
        file_path=out_path,
        split_sentences=True,
    )
    return FileResponse(out_path, media_type='audio/wav', filename='yuki_xtts.wav')

In [ ]:
import subprocess
import time
from pathlib import Path

log_path = Path('/content/xtts_server.log')
with open(log_path, 'w', encoding='utf-8') as log:
    server = subprocess.Popen(
        [PYTHON_BIN, '-m', 'uvicorn', 'xtts_server:app', '--host', '0.0.0.0', '--port', '8020'],
        cwd='/content',
        stdout=log,
        stderr=subprocess.STDOUT,
    )

started = False
for _ in range(180):
    time.sleep(2)
    try:
        import requests
        r = requests.get('http://127.0.0.1:8020/health', timeout=5)
        if r.ok:
            print('Servidor XTTS ativo:', r.json())
            started = True
            break
    except Exception:
        pass
    if server.poll() is not None:
        break

if not started:
    print(log_path.read_text(encoding='utf-8', errors='ignore')[-12000:])
    raise RuntimeError('Servidor XTTS nao iniciou.')

In [ ]:
import requests
r = requests.post(
    'http://127.0.0.1:8020/tts_to_audio',
    json={'text': 'Olá Lucas, agora a Yuki está falando pelo XTTS v2.', 'speaker_wav': '/content/ref_audio/reference.wav', 'language': 'pt'},
    timeout=600,
)
print('warmup status =', r.status_code, 'bytes =', len(r.content))
open('/content/xtts_warmup.wav', 'wb').write(r.content)
print('Warmup concluído.')

In [ ]:
import re
import subprocess
import time
from pathlib import Path

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

log_path = Path('/content/cloudflared_xtts.log')
with open(log_path, 'w') as log:
    tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8020'], stdout=log, stderr=log)

public_url = None
for _ in range(60):
    time.sleep(2)
    text = log_path.read_text(encoding='utf-8', errors='ignore')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError('Nao consegui obter a URL publica do tunnel.')

print('COLE NO conf.yaml:')
print("  tts_model: 'x_tts'")
print(f"  api_url: '{public_url}/tts_to_audio'")
print("  speaker_wav: '/content/ref_audio/reference.wav'")
print("  language: 'pt'")

In [ ]:
import time
while True:
    print('Yuki XTTS online:', time.strftime('%H:%M:%S'))
    time.sleep(60)